In [0]:
from pyspark.sql import functions as F

df_gold = spark.table("teste_koin.default.gold_orders_customers")

df_monthly_revenue = (
    df_gold
    .withColumn(
        "year_month",
        F.date_format("order_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.round(F.sum("order_amount"), 2).alias("total_revenue"),
        F.round(F.avg("order_amount"), 2).alias("average_ticket"),
        F.count("order_id").alias("total_orders")
    )
    .orderBy("year_month")
)

(
    df_monthly_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("teste_koin.default.gold_monthly_revenue")
)